## Macroeconomic Labor Dynamics: The Phillips Curve via Pure SQL

In questo notebook interroghiamo direttamente le API REST di Eurostat (SDMX-CSV) tramite SQL puro utilizzando l'estensione `httpfs` di DuckDB. 

Per evitare i blocchi del server (`HTTP 0 Internal Server Error`), impostiamo un User-Agent personalizzato prima di eseguire le istruzioni di lettura.

In [ ]:
# !pip install duckdb duckdb-engine jupysql matplotlib

import duckdb
import matplotlib.pyplot as plt

# Carichiamo l'estensione JupySQL e inizializziamo DuckDB in memoria
%load_ext sql
%sql duckdb:///:memory:

In [ ]:
%%sql phillips_curve_df <<
INSTALL httpfs;
LOAD httpfs;

-- 1. FONDAMENTALE: Mascheriamo DuckDB come un normale browser web per aggirare il firewall di Eurostat
SET custom_user_agent = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36';

-- Nota: Se sei dietro un proxy aziendale, de-commenta e configura queste due righe:
-- SET http_proxy = 'http://proxy.tuo-dominio.com:8080';
-- SET https_proxy = 'http://proxy.tuo-dominio.com:8080';

-- 2. Estraiamo i dati direttamente dalle API in formato CSV e creiamo la nostra tabella unita
WITH unemployment AS (
    SELECT 
        TIME_PERIOD AS period,
        CAST(OBS_VALUE AS FLOAT) AS unemployment_rate
    FROM read_csv_auto('https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/une_rt_m/M.SA.Y15-74.PC_ACT.T.IT?format=SDMX-CSV&startPeriod=2014-01')
    WHERE OBS_VALUE IS NOT NULL
),
inflation AS (
    SELECT 
        TIME_PERIOD AS period,
        CAST(OBS_VALUE AS FLOAT) AS inflation_rate
    FROM read_csv_auto('https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/prc_hicp_manr/M.RCH_A.CP00.IT?format=SDMX-CSV&startPeriod=2014-01')
    WHERE OBS_VALUE IS NOT NULL
)
SELECT 
    u.period,
    u.unemployment_rate,
    i.inflation_rate
FROM unemployment u
JOIN inflation i 
    ON u.period = i.period
ORDER BY u.period DESC;

In [ ]:
# Verifichiamo che i dati siano stati scaricati correttamente via SQL
df_plot = phillips_curve_df.DataFrame()
display(df_plot.head())

# Disegniamo la Curva di Phillips empirica
plt.figure(figsize=(10, 6))
plt.scatter(
    df_plot['unemployment_rate'], 
    df_plot['inflation_rate'], 
    alpha=0.6, 
    color='darkblue', 
    edgecolors='k'
)

plt.title('Empirical Phillips Curve: Italy (2014 - Present)\n(Data queried directly via SQL from Eurostat API)', fontsize=14, fontweight='bold')
plt.xlabel('Unemployment Rate (%)', fontsize=12)
plt.ylabel('Inflation Rate (HICP % Change)', fontsize=12)

plt.axhline(0, color='black', linewidth=1.2, linestyle='--')
plt.grid(True, linestyle=':', alpha=0.7)
plt.tight_layout()
plt.show()